# 4. BIB checks

I check which unpaired runs have cells and how the jet cuts change the sample.
Unpaired bunch crossings give me data enriched in beam-induced background (BIB).
My signal sample is simulated jets matched to long-lived particles (LLPs).

Run with **PPP first look** and the local ROOT files in `DataForEmre/`.
[Train and compare](03_train_and_compare.ipynb) has the main model comparison.

In [ ]:
from collections import Counter
from pathlib import Path
import json

import awkward as ak
import numpy as np
import pandas as pd
import uproot
from IPython.display import display

project = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "DataForEmre").is_dir() and (p / "notebooks").is_dir())
data = project / "DataForEmre"
outputs = project / "analysis/outputs"

## Which runs have cell data?

I match events by run, event number and luminosity block. The sidecar
is a separate file with primary vertex counts. It stores event IDs as signed
32-bit numbers, so I read the same bits as unsigned before matching.

In [ ]:
unpaired = data / "non_collision/unpaired"
llp_files = sorted((data / "collision/LLP/N1N2_rpv_1400_10ns").glob("*.root"))
cell_files = sorted((unpaired / "physics_Background_NTuple_unsplit_v2").glob("*.root"))
vertex_files = sorted((unpaired / "user.ezaya.UNPAIRED_nPV_nocells.EXT0").glob("*.root"))
assert llp_files and cell_files and vertex_files, "Check the three local ROOT folders."

In [ ]:
cell_keys = set()
cell_runs = Counter()
cell_id = ["runNumber", "eventNumber", "lumiBlock"]
for path in cell_files:
    with uproot.open(path) as source:
        ids = source["JetContent"].arrays(cell_id, library="np")
    run, event, lumi = (ids[field] for field in cell_id)
    keys = set(zip(map(int, run), map(int, event), map(int, lumi)))
    assert len(keys) == len(run) and cell_keys.isdisjoint(keys), "Repeated cell event."
    cell_keys.update(keys)
    cell_runs.update(map(int, run))

In [ ]:
vertex_id = ["runNumber", "eventNumber", "lumiBlockNumber", "nPrimaryVertices"]
sidecar_runs = Counter()
vertices = {}

def add_vertex_chunk(chunk):
    run, event, lumi, count = (chunk[field] for field in vertex_id)
    runs, counts = np.unique(run, return_counts=True)
    sidecar_runs.update(dict(zip(map(int, runs), map(int, counts))))
    needed = np.isin(run, list(cell_runs))
    assert event.dtype == np.dtype("int32"), "Recheck event ID storage before matching."
    event = event.astype(np.uint32)
    for r, e, block, n in zip(run[needed], event[needed], lumi[needed], count[needed]):
        key = (int(r), int(e), int(block))
        if key in cell_keys:
            assert key not in vertices and int(n) >= 0, f"Invalid vertex match {key}"
            vertices[key] = int(n)

In [ ]:
for path in vertex_files:
    with uproot.open(path) as source:
        for chunk in source["BIB"].iterate(vertex_id, step_size=100_000, library="np"):
            add_vertex_chunk(chunk)

runs = sorted(cell_runs.keys() | sidecar_runs.keys())
run_table = pd.DataFrame({"Run": runs,
    "Events with cells": [cell_runs[run] for run in runs],
    "Sidecar rows": [sidecar_runs[run] for run in runs],
}).set_index("Run")
display(run_table)
assert cell_keys == vertices.keys(), "Some local cell events have no valid vertex match."

For my local files, run **484258** has cells. Run **486706** has sidecar
data, so I can check jet variables there. I would need its cell export
to test ParT on that run.

## How many jets pass the cuts?

I keep LLP-matched simulation jets and unpaired jets from events with
zero reconstructed primary vertices. Both samples use $|\eta|<1.7$,
where $\eta$ describes direction relative to the beam. I lower the
transverse momentum cut from 30 to 15 GeV, then drop the requirement
that both calorimeters contribute ($0<f_{\rm Tile}<1$).
Stored jet time above 3 ns counts as delayed.

In [ ]:
jet_fields = ["jet_pt", "jet_eta", "jet_time_ATLAS", "jet_TileEnergyFrac"]
jet_selections = [(30, True), (15, True), (15, False)]
selection_names = ["30 GeV, mixed", "15 GeV, mixed", "15 GeV, all calorimeters"]
region_names = ["All times", "Time > 3 ns", "3 < time ≤ 10 ns"]

def valid(values):
    return np.isfinite(values) & (values != -999999)

def vertex_mask(rows):
    keys = zip(*(map(int, ak.to_numpy(rows[field])) for field in cell_id))
    return ak.Array([vertices[key] == 0 for key in keys])

In [ ]:
def masks(rows, sample):
    pt, eta, time, tile = (rows[field] for field in jet_fields)
    valid_jet = valid(pt) & valid(eta) & valid(time) & (abs(eta) < 1.7)
    if sample == "LLP":
        valid_jet = valid_jet & (rows["jet_is_llp_matched"] == 1)
    else:
        valid_jet = valid_jet & vertex_mask(rows)
    mixed = (tile > 0) & (tile < 1)
    inclusive = (tile >= 0) & (tile <= 1)
    for min_pt, require_mixed in jet_selections:
        calorimeters = mixed if require_mixed else inclusive
        keep = valid_jet & (pt >= 1000 * min_pt) & calorimeters
        delayed = keep & (time > 3)
        yield keep, delayed, delayed & (time <= 10)

In [ ]:
def count_sample(paths, sample):
    fields = jet_fields + (cell_id if sample == "unpaired" else ["jet_is_llp_matched"])
    counts = np.zeros((len(jet_selections), len(region_names)), dtype=np.int64)
    for path in paths:
        with uproot.open(path) as source:
            batches = source["JetContent"].iterate(fields, step_size=10_000, library="ak")
            for rows in batches:
                for i, selections in enumerate(masks(rows, sample)):
                    counts[i] += [int(ak.sum(keep)) for keep in selections]
    return counts

In [ ]:
selection_counts = {}
for sample, paths in [("LLP", llp_files), ("unpaired", cell_files)]:
    counts = count_sample(paths, sample)
    table = pd.DataFrame(counts, index=selection_names, columns=region_names)
    table["Delayed (%)"] = (100 * table["Time > 3 ns"] / table["All times"]).round(2)
    assert (table["All times"] >= table["Time > 3 ns"]).all()
    assert (table["Time > 3 ns"] >= table["3 < time ≤ 10 ns"]).all()
    assert (np.diff(counts, axis=0) >= 0).all()
    print(sample)
    display(table)
    selection_counts[sample] = table

Lowering the momentum cut gives me 326 delayed unpaired jets instead
of 240. Including jets with $f_{\rm Tile}=0$ or $1$ raises that to
11,889. The delayed LLP count barely changes.

## Two earlier checks

These are optional reports from local saved runs. The run comparison
uses jet variables and a cut set to keep 95% of delayed LLP jets on
validation. Its test sample and split belong to that earlier study.

In [ ]:
def saved_file(study, name):
    path = outputs / study / name
    if path.is_file():
        return path
    print(f"Local report unavailable: {path.relative_to(project)}")
    return None

def ratio(success, total):
    return f"{int(success):,} / {int(total):,}"

In [ ]:
path = saved_file("unpaired_run_study", "results.csv")
if path is not None:
    report = pd.read_csv(path)
    delayed = report.query("region == 'delayed' and validation_region == 'delayed'").copy()
    delayed["Unpaired tagged / available"] = [ratio(hit, total) for hit, total in
        zip(delayed["unpaired_tagged"], delayed["unpaired_test_jets"])]
    labels = {"time_only": "Time only", "kinematics_only": "Kinematics only",
              "time_and_kinematics": "Time and kinematics"}
    display(delayed.pivot(index="run", columns="model", values="Unpaired tagged / available")
            .rename(columns=labels).reindex(columns=labels.values()))
    llp = delayed[["model", "llp_kept", "llp_test_jets"]].drop_duplicates().set_index("model")
    print("The same delayed LLP test jets were scored for every run.")
    display(llp.rename(index=labels, columns={"llp_kept": "LLP kept", "llp_test_jets": "LLP tested"}))

In the cell check I shuffle times within each jet, $t_i\mapsto t_{\pi(i)}$,
so the time values stay the same but attach to different positions.
I keep ParT's original model and cut. I also compare against a score
using only $-N_{\rm cells}$ with its own validation cut.

In [ ]:
path = saved_file("cell_input_study", "metrics.json")
if path is not None:
    report = json.loads(path.read_text())
    print("Median test cell counts", report["median_cell_count_test"])
    rows = []
    for method, stats in report["comparisons"]["delayed"].items():
        hits, totals = stats["confusion"], stats["class_counts"]
        rows.append({"Method": method.replace("_", " "),
                     "Unpaired tagged / total": ratio(hits["BIB_tagged_BIB"], totals["BIB"]),
                     "LLP kept / total": ratio(hits["LLP_kept_as_LLP"], totals["LLP"])})
    display(pd.DataFrame(rows).set_index("Method"))

In that check, the shuffle reduces LLP retention and the count-only
score tags most unpaired jets. Both effects are measured on the same
small delayed sample, so I'd check them again with cells from another run.